# **Berlin Restaurant Market Analysis**
What goes along with good ratings and lots of reviews in Berlin's restaurant scene

Tools: Python · Pandas · Matplotlib · Seaborn · SciPy

Dataset: TripAdvisor European Restaurants (Kaggle)

Author: Siddharth Suman

> **Update, October 2026.** I went back through this notebook and fixed a few things. The claimed vs unclaimed rating gap is 0.14 stars (about 3.5%), not 14% as I first wrote. The Berlin filter was also picking up a town called Berlingen. And I had filled half the missing sub-ratings with the median before running correlations, which waters them down. Details are in the cells where each fix happens.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# I ran this in Google Colab with the CSV in my Drive.
# On your own machine: download the Kaggle file (link in the README)
# and put it in a folder called data/ next to this notebook.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/tripadvisor_european_restaurants.csv'
except ImportError:
    DATA_PATH = 'data/tripadvisor_european_restaurants.csv'

os.makedirs('images', exist_ok=True)

df = pd.read_csv(DATA_PATH, low_memory=False)
print(df.shape)
df.head()

In [ ]:
# First look at the data
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
df.head()


In [ ]:
germany = df[df['country'] == 'Germany'].copy()
print("Germany restaurants:", len(germany))

# \b means "word boundary". Without it, 'Berlin' also matches 'Berlingen',
# which is a different town. I missed that in the first version.
pattern = r'\bBerlin\b'
berlin = germany[
    germany['city'].str.contains(pattern, case=False, na=False, regex=True) |
    germany['province'].str.contains(pattern, case=False, na=False, regex=True) |
    germany['region'].str.contains(pattern, case=False, na=False, regex=True)
].copy()

print("Berlin restaurants:", len(berlin))

cols = ['restaurant_name', 'city', 'address', 'latitude', 'longitude',
        'claimed', 'awards', 'price_level', 'cuisines', 'meals',
        'vegetarian_friendly', 'vegan_options', 'avg_rating',
        'total_reviews_count', 'excellent', 'very_good', 'average',
        'poor', 'terrible', 'food', 'service', 'value', 'atmosphere',
        'top_tags', 'open_days_per_week', 'open_hours_per_week']

berlin = berlin[cols]
print("Berlin dataset shape:", berlin.shape)
berlin.head()

In [ ]:
#DATA CLEANING

# 1. Check missing values in our Berlin dataset
print("Missing values:\n")
print(berlin.isnull().sum())

In [ ]:
# Restaurants with no rating can't tell us anything here
berlin = berlin.dropna(subset=['avg_rating'])
print("After dropping missing ratings:", len(berlin))

berlin['price_level'] = berlin['price_level'].fillna('Unknown')
berlin['claimed'] = berlin['claimed'].fillna('Unclaimed')

# In the first version I filled missing food/service/value/atmosphere scores
# with the median. About half of them are missing, so that put thousands of
# identical fake values into the data and weakened every correlation.
# Now I leave them as NaN. pandas .corr() simply skips missing pairs.
berlin = berlin.reset_index(drop=True)

print("Clean dataset shape:", berlin.shape)
print("\nHow many restaurants actually have each sub-rating:")
print(berlin[['food', 'service', 'value', 'atmosphere', 'open_hours_per_week']].notna().sum())

In [ ]:
# EDA: OVERVIEW STATISTICS
print("=" * 50)
print("BERLIN RESTAURANT DATASET — KEY STATS")
print("=" * 50)

print(f"\nTotal restaurants analysed: {len(berlin):,}")
print(f"Average rating: {berlin['avg_rating'].mean():.2f}")
print(f"Median rating: {berlin['avg_rating'].median():.2f}")
print(f"Highest rated: {berlin['avg_rating'].max()}")
print(f"Lowest rated: {berlin['avg_rating'].min()}")
print(f"\nAverage reviews per restaurant: {berlin['total_reviews_count'].mean():.0f}")
print(f"Most reviewed restaurant: {berlin.loc[berlin['total_reviews_count'].idxmax(), 'restaurant_name']} ({berlin['total_reviews_count'].max():.0f} reviews)")
print(f"\nPrice level distribution:\n{berlin['price_level'].value_counts()}")
print(f"\nClaimed vs Unclaimed:\n{berlin['claimed'].value_counts()}")

## Key Finding 1: Berlin Restaurant Landscape

Berlin's 6,594 restaurants maintain a strong average rating of 4.15/5, with the bulk of establishments concentrated in the mid-range price segment (€€-€€€). This points to a fiercely competitive middle market, one where standing out demands more than a decent menu. Digital presence, service consistency and value perception are what separate the memorable from the merely adequate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Berlin Restaurant Ratings Overview', fontsize=14, fontweight='bold')
axes[0].hist(berlin['avg_rating'], bins=30, color='#1F4E79', edgecolor='white')
axes[0].set_title('Distribution of Average Ratings')
axes[0].set_xlabel('Average Rating')
axes[0].set_ylabel('Number of Restaurants')
axes[0].axvline(berlin['avg_rating'].mean(), color='red', linestyle='--',
                label=f"Mean: {berlin['avg_rating'].mean():.2f}")
axes[0].legend()
price_counts = berlin['price_level'].value_counts()
axes[1].bar(price_counts.index, price_counts.values, color='#1F4E79', edgecolor='white')
axes[1].set_title('Restaurants by Price Level')
axes[1].set_xlabel('Price Level')
axes[1].set_ylabel('Number of Restaurants')
plt.tight_layout()
plt.savefig('images/1_ratings_overview.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# EDA: INSIGHT 2 : Does being CLAIMED affect ratings?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Claimed vs Unclaimed Restaurants in Berlin',
             fontsize=14, fontweight='bold')

# Average rating by claimed status
claimed_rating = berlin.groupby('claimed')['avg_rating'].mean()
axes[0].bar(claimed_rating.index, claimed_rating.values,
            color=['#1F4E79', '#C44B4B'], edgecolor='white')
axes[0].set_title('Average Rating: Claimed vs Unclaimed')
axes[0].set_xlabel('Status')
axes[0].set_ylabel('Average Rating')
axes[0].set_ylim(3.5, 4.5)
for i, v in enumerate(claimed_rating.values):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

# Average reviews by claimed status
claimed_reviews = berlin.groupby('claimed')['total_reviews_count'].mean()
axes[1].bar(claimed_reviews.index, claimed_reviews.values,
            color=['#1F4E79', '#C44B4B'], edgecolor='white')
axes[1].set_title('Average Reviews: Claimed vs Unclaimed')
axes[1].set_xlabel('Status')
axes[1].set_ylabel('Average Review Count')
for i, v in enumerate(claimed_reviews.values):
    axes[1].text(i, v + 1, f'{v:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('images/2_claimed_vs_unclaimed.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the numbers
print("Average rating by status:")
print(claimed_rating)
print("\nAverage reviews by status:")
print(claimed_reviews)

## Key Finding 2: The Digital Presence Gap

Claimed restaurants (the owner has taken control of the TripAdvisor page) do better on both measures:

- **0.14 stars higher on average** (4.25 vs 4.10, roughly 3.5%)
- **3.4x more reviews** (about 162 vs 48)

The review gap is the big one. The rating gap is real but small.

Two thirds of Berlin restaurants (about 4,350) haven't claimed their page. That's a lot of restaurants that aren't managing how they look online, and a delivery platform could help with that.

One caution: this doesn't show that claiming a page *causes* better results. Busy, popular restaurants are probably more likely to bother claiming their page in the first place.

In [ ]:
# EDA: INSIGHT 3 — Price level vs Rating
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Price Level vs Performance in Berlin',
             fontsize=14, fontweight='bold')

# Filter out Unknown price
price_data = berlin[berlin['price_level'] != 'Unknown']

# Rating by price level
price_order = ['€', '€€-€€€', '€€€€']
price_rating = price_data.groupby('price_level')['avg_rating'].mean()[price_order]
axes[0].bar(price_rating.index, price_rating.values,
            color=['#2ecc71', '#1F4E79', '#C44B4B'], edgecolor='white')
axes[0].set_title('Average Rating by Price Level')
axes[0].set_xlabel('Price Level')
axes[0].set_ylabel('Average Rating')
axes[0].set_ylim(3.5, 4.5)
for i, v in enumerate(price_rating.values):
    axes[0].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

# Reviews by price level
price_reviews = price_data.groupby('price_level')['total_reviews_count'].mean()[price_order]
axes[1].bar(price_reviews.index, price_reviews.values,
            color=['#2ecc71', '#1F4E79', '#C44B4B'], edgecolor='white')
axes[1].set_title('Average Reviews by Price Level')
axes[1].set_xlabel('Price Level')
axes[1].set_ylabel('Average Review Count')
for i, v in enumerate(price_reviews.values):
    axes[1].text(i, v + 1, f'{v:.0f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('images/3_price_vs_performance.png', dpi=150, bbox_inches='tight')
plt.show()

print("Rating by price level:")
print(price_rating)
print("\nReviews by price level:")
print(price_reviews)

## Key Finding 3: The Price vs Performance Paradox

Counterintuitively, budget restaurants (€) and fine dining (€€€€) both outrate mid-range establishments. Fine dining attracts 4x more reviews than budget options, suggesting considerably deeper customer engagement at premium price points.

**Business Recommendation:** Mid-range restaurants face the steepest competition in Berlin. Delivery platforms targeting this segment should prioritise review generation strategies to help partners carve out a distinct identity in an overcrowded market.

In [ ]:
# Top 10 cuisine types in Berlin
# Each restaurant lists several cuisines ("Italian, Pizza, European"), so I
# split them into one row per cuisine. I match each cuisine exactly. My first
# version used str.contains, so 'European' also counted 'Central European',
# and 'Bar' also counted 'Barbecue'.

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Cuisine Analysis in Berlin', fontsize=14, fontweight='bold')

cuisine_long = (berlin[['restaurant_name', 'avg_rating', 'cuisines']]
                .dropna(subset=['cuisines'])
                .assign(cuisine=lambda d: d['cuisines'].str.split(', '))
                .explode('cuisine'))
cuisine_long['cuisine'] = cuisine_long['cuisine'].str.strip()

top_cuisines = cuisine_long['cuisine'].value_counts().head(10)

axes[0].barh(top_cuisines.index[::-1], top_cuisines.values[::-1],
             color='#1F4E79', edgecolor='white')
axes[0].set_title('Top 10 Cuisines by Restaurant Count')
axes[0].set_xlabel('Number of Restaurants')
for i, v in enumerate(top_cuisines.values[::-1]):
    axes[0].text(v + 5, i, str(v), va='center', fontweight='bold')

cuisine_rating_df = (cuisine_long[cuisine_long['cuisine'].isin(top_cuisines.index)]
                     .groupby('cuisine')['avg_rating'].mean()
                     .sort_values()
                     .reset_index())

axes[1].barh(cuisine_rating_df['cuisine'], cuisine_rating_df['avg_rating'],
             color='#2ecc71', edgecolor='white')
axes[1].set_title('Average Rating by Top Cuisine')
axes[1].set_xlabel('Average Rating')
axes[1].set_xlim(3.8, 4.4)
for i, v in enumerate(cuisine_rating_df['avg_rating']):
    axes[1].text(v + 0.005, i, f'{v:.2f}', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('images/4_cuisines.png', dpi=150, bbox_inches='tight')
plt.show()

print("Top 10 cuisines:")
print(top_cuisines)
print("\nRatings by cuisine:")
print(cuisine_rating_df.to_string(index=False))

## Key Finding 4: Cuisine Landscape

European, café and German places are the most common in Berlin. The rating chart tells a slightly different story: cafés, Mediterranean and Asian restaurants rate higher on average than German and pizza places, even though there are fewer of them.

The gaps between cuisines are small, mostly a tenth of a star or two, so I wouldn't read too much into the exact order. What stands out is that the most common types aren't the best-rated ones. For a delivery platform, Asian and Mediterranean restaurants look like a good group to sign up: plenty of them, and customers like them.

In [ ]:
# What goes along with a high rating? Correlation matrix.
# Missing values are skipped pair by pair, so each number is based only on
# restaurants that actually have both scores.
fig, ax = plt.subplots(figsize=(10, 7))

corr_cols = ['avg_rating', 'food', 'service', 'value',
             'atmosphere', 'total_reviews_count',
             'open_days_per_week', 'open_hours_per_week']

corr_matrix = berlin[corr_cols].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Correlation Matrix: What Goes With Higher Ratings in Berlin?',
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('images/5_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Correlation with avg_rating:")
print(corr_matrix['avg_rating'].sort_values(ascending=False).round(2))

## Key Finding 5: What goes along with good ratings

Service, food and value all line up with the overall rating about equally (see the numbers printed above). In the first version I said service was the strongest predictor. The three are too close together to pick a winner, and they're all scores from the same reviewers, so it's no surprise they move together.

Atmosphere has a much weaker link than the other three. A simple place with good service and fair prices seems to do just as well as a beautiful one.

Restaurants with longer opening hours tend to rate a little lower. That's a weak link though, and it could easily be down to the type of place (late-night fast food, for example) rather than the hours themselves.

**For a delivery platform:** if you're coaching restaurant partners, service and value for money are better places to start than decor or longer hours.

In [ ]:
# Claimed vs unclaimed: is the rating gap bigger than chance?
# This is an observational comparison, not an A/B test. Nobody was randomly
# assigned to claim their page, so this can show a difference but not a cause.

claimed = berlin.loc[berlin['claimed'] == 'Claimed', 'avg_rating']
unclaimed = berlin.loc[berlin['claimed'] == 'Unclaimed', 'avg_rating']

# The two groups have quite different spreads, so I use Welch's t-test,
# which doesn't assume equal variances.
t_stat, p_value = stats.ttest_ind(claimed, unclaimed, equal_var=False)

# p-values get tiny with thousands of rows, so also report how BIG the gap is.
diff = claimed.mean() - unclaimed.mean()
pooled_sd = np.sqrt(((len(claimed) - 1) * claimed.var() + (len(unclaimed) - 1) * unclaimed.var())
                    / (len(claimed) + len(unclaimed) - 2))
cohens_d = diff / pooled_sd
se = np.sqrt(claimed.var() / len(claimed) + unclaimed.var() / len(unclaimed))
ci_low, ci_high = diff - 1.96 * se, diff + 1.96 * se

print("Claimed vs Unclaimed: average rating")
print("=" * 52)
print(f"Claimed:   n={len(claimed):,} | mean={claimed.mean():.3f} | std={claimed.std():.3f}")
print(f"Unclaimed: n={len(unclaimed):,} | mean={unclaimed.mean():.3f} | std={unclaimed.std():.3f}")
print(f"\nDifference: {diff:.2f} stars ({diff / unclaimed.mean():.1%})")
print(f"95% CI for the difference: {ci_low:.2f} to {ci_high:.2f} stars")
print(f"Cohen's d: {cohens_d:.2f}  (around 0.2 is usually called a small effect)")
print(f"Welch t = {t_stat:.2f}, p = {p_value:.1e}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Claimed vs Unclaimed: How Big Is the Rating Gap?', fontsize=13, fontweight='bold')

axes[0].hist(unclaimed, bins=30, alpha=0.6, color='#C44B4B',
             label=f'Unclaimed (n={len(unclaimed):,})', edgecolor='white')
axes[0].hist(claimed, bins=30, alpha=0.6, color='#1F4E79',
             label=f'Claimed (n={len(claimed):,})', edgecolor='white')
axes[0].axvline(claimed.mean(), color='#1F4E79', linestyle='--', linewidth=2)
axes[0].axvline(unclaimed.mean(), color='#C44B4B', linestyle='--', linewidth=2)
axes[0].set_title('The two distributions overlap a lot')
axes[0].set_xlabel('Average Rating')
axes[0].set_ylabel('Number of Restaurants')
axes[0].legend()

axes[1].boxplot([unclaimed, claimed], tick_labels=['Unclaimed', 'Claimed'],
                patch_artist=True,
                boxprops=dict(facecolor='#E8EEF4'),
                medianprops=dict(color='#1F4E79', linewidth=2))
axes[1].set_title(f'Gap: {diff:.2f} stars, Cohen\'s d = {cohens_d:.2f}')
axes[1].set_ylabel('Average Rating')

plt.tight_layout()
plt.savefig('images/6_claimed_rating_test.png', dpi=150, bbox_inches='tight')
plt.show()

## Test result: a real gap, but a small one

**Question:** Do restaurants that have claimed their TripAdvisor page have higher ratings than those that haven't?

**Answer:** Yes, but not by much. Claimed restaurants average about 0.14 stars more, and the 95% confidence interval stays well clear of zero. With over 6,500 restaurants, the p-value is tiny. That only tells us the gap is unlikely to be chance, not that it's big. The effect size (Cohen's d around 0.2) says it's small, and the histograms overlap almost completely.

I first called this an A/B test. It isn't one. Nobody was randomly told to claim their page, so I can't say claiming *causes* the higher rating. It could just as well be that well-run, popular restaurants are the ones that bother to claim their page.

**What a platform could actually do with this:** the review gap (3.4x) matters more than the rating gap. If a platform wanted to know whether helping restaurants claim and manage their profiles really works, the honest way to find out is a small randomised pilot: offer onboarding support to a random half of unclaimed partners and compare them after a few months.

In [ ]:
# Top performing restaurants in Berlin
# Lots of places are tied at 5.0, so I break ties by review count.
# Otherwise the top 10 is just whichever 5.0 places happen to come first.
top_restaurants = berlin[berlin['total_reviews_count'] >= 100].copy()

top_rated = (top_restaurants
             .sort_values(['avg_rating', 'total_reviews_count'], ascending=False)
             .head(10)[['restaurant_name', 'cuisines',
                        'price_level', 'avg_rating',
                        'total_reviews_count', 'claimed']])

most_reviewed = (berlin
                 .sort_values('total_reviews_count', ascending=False)
                 .head(10)[['restaurant_name', 'cuisines',
                            'price_level', 'avg_rating',
                            'total_reviews_count', 'claimed']])

print("TOP 10 HIGHEST RATED (min 100 reviews)")
print("=" * 60)
print(top_rated.to_string(index=False))

print("\nTOP 10 MOST REVIEWED IN BERLIN")
print("=" * 60)
print(most_reviewed.to_string(index=False))

# final visual — top 10 most reviewed
fig, ax = plt.subplots(figsize=(12, 6))
most_reviewed_plot = most_reviewed.sort_values('total_reviews_count')
bars = ax.barh(most_reviewed_plot['restaurant_name'],
               most_reviewed_plot['total_reviews_count'],
               color='#1F4E79', edgecolor='white')
ax.set_title("Berlin's 10 Most Reviewed Restaurants on TripAdvisor",
             fontsize=13, fontweight='bold')
ax.set_xlabel('Total Reviews')
for bar, val in zip(bars, most_reviewed_plot['total_reviews_count']):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('images/7_top_restaurants.png', dpi=150, bbox_inches='tight')
plt.show()

perfect = top_restaurants[top_restaurants['avg_rating'] == 5.0]
print(f"\nRestaurants rated 5.0 with 100+ reviews: {len(perfect)}, "
      f"of which claimed: {(perfect['claimed'] == 'Claimed').sum()}")

## Key Finding 6: What Berlin's best restaurants have in common

Every restaurant in this top 10 is claimed. Combined with the review gap above, it suggests that the restaurants doing best are also the ones paying attention to their online presence. (Again, which comes first is an open question.)

The top 10 is a real mix: burgers, Indian, Italian, Middle Eastern, German. No single cuisine dominates. In the first version I said Indian restaurants dominate the perfect-score list. With two out of ten, that was an overstatement.

The most-reviewed list has a very Berlin entry: Mustafa's Gemuese Kebab, a cheap street food stand, is fourth in the whole city with over 4,300 reviews and a 4.5 rating. Berlin clearly doesn't need white tablecloths to rate a place highly.

In [ ]:
# SUMMARY VISUALISATION
# The main findings in one view

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Berlin Restaurant Market: Key Insights',
             fontsize=16, fontweight='bold', y=1.01)

# 1. Rating distribution by price
price_order = ['€', '€€-€€€', '€€€€']
price_data = berlin[berlin['price_level'].isin(price_order)]
price_box_data = [price_data[price_data['price_level'] == p]['avg_rating'].values
                  for p in price_order]
axes[0,0].boxplot(price_box_data, tick_labels=price_order,
                  patch_artist=True,
                  boxprops=dict(facecolor='#E8EEF4'),
                  medianprops=dict(color='#1F4E79', linewidth=2))
axes[0,0].set_title('Rating Distribution by Price Level')
axes[0,0].set_ylabel('Average Rating')

# 2. Claimed vs unclaimed reviews
claimed_reviews = berlin.groupby('claimed')['total_reviews_count'].mean()
axes[0,1].bar(claimed_reviews.index, claimed_reviews.values,
              color=['#1F4E79', '#C44B4B'], edgecolor='white')
axes[0,1].set_title('Average Reviews: Claimed vs Unclaimed')
axes[0,1].set_ylabel('Average Review Count')
for i, v in enumerate(claimed_reviews.values):
    axes[0,1].text(i, v + 2, f'{v:.0f}', ha='center', fontweight='bold')

# 3. Top 5 cuisines by avg rating
top5_cuisine_rating = cuisine_rating_df.tail(5)
axes[1,0].barh(top5_cuisine_rating['cuisine'],
               top5_cuisine_rating['avg_rating'],
               color='#2ecc71', edgecolor='white')
axes[1,0].set_title('Top 5 Cuisines by Average Rating')
axes[1,0].set_xlabel('Average Rating')
axes[1,0].set_xlim(3.8, 4.4)
for i, v in enumerate(top5_cuisine_rating['avg_rating']):
    axes[1,0].text(v + 0.01, i, f'{v:.2f}', va='center', fontweight='bold')

# 4. What drives ratings — top correlations
driver_corr = corr_matrix['avg_rating'].drop('avg_rating').sort_values()
colors = ['#C44B4B' if v < 0 else '#1F4E79' for v in driver_corr.values]
axes[1,1].barh(driver_corr.index, driver_corr.values,
               color=colors, edgecolor='white')
axes[1,1].set_title('What Goes With Higher Ratings')
axes[1,1].set_xlabel('Correlation with Average Rating')
axes[1,1].axvline(0, color='black', linewidth=0.8)
for i, v in enumerate(driver_corr.values):
    axes[1,1].text(v + 0.005 if v >= 0 else v - 0.005, i,
                   f'{v:.2f}', va='center', ha='left' if v >= 0 else 'right',
                   fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('images/8_summary.png', dpi=150, bbox_inches='tight')
plt.show()


## Limitations and what I'd do next

- **These are correlations.** Nothing here was randomised, so none of it proves cause and effect.
- **It's TripAdvisor.** TripAdvisor reviewers lean towards tourists, so a place that locals love might not show up well.
- **Ratings are rounded to half stars,** which squashes a lot of the variation.
- **Next steps:** control for price level and cuisine in a regression to see whether the claimed gap holds up within similar restaurants, and map ratings by district using the latitude and longitude columns.

In [ ]:
# Colab only: download all charts in one zip so they can be uploaded to GitHub
try:
    from google.colab import files
    import shutil
    shutil.make_archive('images', 'zip', 'images')
    files.download('images.zip')
except ImportError:
    print("Not in Colab. The charts are in the images/ folder.")